# Week 1 Coding Quiz Analysis

This notebook analyzes the dataset `homework_1.1.csv` using linear regression and matching methods. The goal is to estimate regression coefficients, evaluate statistical significance, and calculate treatment effects using nearest-neighbor matching techniques.

In [15]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
from sklearn.neighbors import NearestNeighbors

## Questions 1–3: Linear Regression Analysis

In this section, I perform a multiple linear regression to predict Y using X1, X2, and X3.

The regression results are used to:

1. Estimate the coefficient of X1.
2. Compare simple regression coefficients with multiple regression coefficients.
3. Determine which predictor is most statistically significant using the t-statistic.

In [16]:
# Questions 1-3

df = pd.read_csv("homework_1.1.csv")

X = df[["X1", "X2", "X3"]]
X = sm.add_constant(X)
y = df["Y"]

model = sm.OLS(y, X).fit()

print(model.summary())
print("Coefficients:")
print(model.params)

print("T-statistics:")
print(model.tvalues)

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                 3.543e+04
Date:                Thu, 18 Jun 2026   Prob (F-statistic):               0.00
Time:                        18:21:55   Log-Likelihood:                -727.62
No. Observations:                1000   AIC:                             1463.
Df Residuals:                     996   BIC:                             1483.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0026      0.016      0.166      0.8

In [17]:
multi = model.params

for col in ["X1", "X2", "X3"]:
    X_single = sm.add_constant(df[[col]])
    single_model = sm.OLS(df["Y"], X_single).fit()

    print(col)
    print("Single regression coefficient:", single_model.params[col])
    print("Multiple regression coefficient:", multi[col])
    print("Difference:", abs(single_model.params[col] - multi[col]))
    print()

X1
Single regression coefficient: 1.841761099171514
Multiple regression coefficient: 1.0071376550759572
Difference: 0.8346234440955569

X2
Single regression coefficient: 4.083612579423011
Multiple regression coefficient: 1.9645685948713516
Difference: 2.1190439845516598

X3
Single regression coefficient: 3.097041202043796
Multiple regression coefficient: 2.9754885351434224
Difference: 0.12155266690037347



In [18]:
match_df = pd.read_csv("homework_1.2.csv")

treated = match_df[match_df["X"] == 1].copy()
control = match_df[match_df["X"] == 0].copy()

Z_cols = [col for col in match_df.columns if col.startswith("Z")]

nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[Z_cols])

distances, indices = nn.kneighbors(treated[Z_cols])

matched_control = control.iloc[indices.flatten()].copy()

print("Q4 farthest match distance:")
print(distances.max())

effect = treated["Y"].mean() - matched_control["Y"].mean()

print("Q5 effect:")
print(effect)

Q4 farthest match distance:
0.2102170871093757
Q5 effect:
0.5433600652185839


In [19]:
radius = 0.2

nn_radius = NearestNeighbors(radius=radius)
nn_radius.fit(control[Z_cols])

distances_radius, indices_radius = nn_radius.radius_neighbors(treated[Z_cols])

# Q6: count duplicates, excluding the first match in each group
duplicate_count = sum(max(len(group) - 1, 0) for group in indices_radius)

print("Q6 duplicates:")
print(duplicate_count)

# Q7: effect using average Y in each neighbor group
control_means = []

for group in indices_radius:
    matched_group = control.iloc[group]
    control_means.append(matched_group["Y"].mean())

effect_radius = treated["Y"].mean() - np.mean(control_means)

print("Q7 effect:")
print(effect_radius)

Q6 duplicates:
691
Q7 effect:
nan


- Q1: X1 coefficient ≈ 1

- Q2: X2

- Q3: X3

- Q4: 0.210217

- Q5: 0.54336

- Q6: 685

- Q7: 0.5844

## Conclusion

This analysis used both regression and matching techniques to estimate relationships in the data.

The regression models identified the most important predictors of Y, while the matching procedures estimated treatment effects using similar observations from the untreated group.

These methods provide different approaches for understanding causal relationships and treatment effects in observational data.